In [ ]:
#| hide
from drona.core import *
from drona.rounds import *
from aidialog.ipynb import read_ipynb, write_ipynb

# drona

> Train agents to choose and use the right tools.

An agent imitates the routes its context shows it. Drona curates that context: it reads what
Ramabana, Claude Code, and Codex leave behind, scores the tool routes, and turns one reviewed
conversation into the opening history of the next session.

```sh
pip install drona
```

## 1. What a session log looks like

Ramabana appends one JSON line per finished turn to `~/.config/ramabana/agent-history.jsonl`. This
session has two turns: the agent flailed, then got it right.

In [ ]:
import json, tempfile
from pathlib import Path

ASK = 'Use fossick to research the AnswerDotAI llmdojo github repository'
detour = {'session': 'sess-9f2a', 'state': 'complete', 'prompt': ASK,
          'reply': 'That search was not useful.',
          'activity': [
              {'action_id': 'a0', 'tool': 'web_search', 'ok': True,
               'args': {'query': 'llmdojo answerdotai'}, 'detail': '10 results, mostly forks.'},
              {'action_id': 'a1', 'tool': 'read_url', 'ok': False,
               'args': {'url': 'https://llmdojo.dev'}, 'detail': '404'}]}
good = {'session': 'sess-9f2a', 'state': 'complete', 'prompt': ASK,
        'reply': 'FOSSICK read the repository directly, so its files are the evidence.',
        'activity': [
            {'action_id': 'b0', 'tool': 'run_shell', 'ok': True,
             'args': {'command': 'fossick read-gh-repo https://github.com/AnswerDotAI/llmdojo'},
             'detail': '# llmdojo\nLLM coding agents imitate what their context shows.'}]}

tmp = Path(tempfile.mkdtemp())
archive = tmp/'agent-history.jsonl'
archive.write_text('\n'.join(json.dumps(t) for t in (detour, good)) + '\n')
print(archive.read_text()[:120], '…')

{"session": "sess-9f2a", "state": "complete", "prompt": "Use fossick to research the AnswerDotAI llmdojo github reposito …


`state` is why Drona reads this file instead of watching a live process. Ramabana marks a turn
`complete`, `failed`, or `abandoned`, and only a complete turn is a route worth imitating.

## 2. Score the routes

`assess_turn` reads what the agent did, never what it said about it, so every finding names a
recorded call by index.

In [ ]:
assess_turn(detour)

Assessment(score=80, calls=2, findings=(Finding(kind='route', tool='web_search', index=0, message='Use fossick read-gh-repo as the first repository research call.'),))

The prompt named a repository and named the tool that reads one, so opening with `web_search` costs
twenty points and the dead `read_url` costs nothing extra — it is the route that is wrong, not the
404. The second turn takes the intended route.

In [ ]:
assess_turn(good)

Assessment(score=100, calls=1, findings=())

```sh
drona --session latest
```

## 3. Capture the session for review

`capture` turns the log into an Aidialog notebook: a skipped review note, then one prompt message
per turn carrying that turn's tool calls and results. The score rides in the metadata.

In [ ]:
review = capture(tmp/'llmdojo.ipynb', history=archive)
dlg = read_ipynb(review)
print(dlg.meta['drona']['score'], dlg.meta['drona']['status'])
for m in dlg: print(f'{m.msg_type:7} skipped={m.skipped}  {str(m.content)[:52]!r}')

80 review
note    skipped=1  '# Drona review\n\nEdit this dialog in Leela. Delete th'
prompt  skipped=0  'Use fossick to research the AnswerDotAI llmdojo gith'
prompt  skipped=0  'Use fossick to research the AnswerDotAI llmdojo gith'


## 4. Review it

This is the whole point of Drona, and it is a person's job. Open the notebook in Leela, delete the
detours and anything private, and keep the route a later model should imitate.

```sh
leela training
```

Leela marks a message skipped when you drop it. Here that is the first turn, the one the assessment
flagged.

In [ ]:
dlg[1].skipped = 1
write_ipynb(dlg, review)
for m in read_ipynb(review): print(f"{m.msg_type:7} {('kept', 'dropped')[m.skipped]:8} {str(m.content)[:44]!r}")

note    dropped  '# Drona review\n\nEdit this dialog in Leela. D'
prompt  dropped  'Use fossick to research the AnswerDotAI llmd'
prompt  kept     'Use fossick to research the AnswerDotAI llmd'


## 5. Accept it

Acceptance is explicit and attributed. It marks the notebook and writes the compiled history beside
it. Nothing compiles, for any host, until this has happened.

In [ ]:
compiled = accept(review, 'Karthik')
meta = json.loads(compiled.read_text())['meta']
print({k: meta[k] for k in ('status', 'reviewer', 'accepted_version')})

{'status': 'accepted', 'reviewer': 'Karthik', 'accepted_version': '0.0.1:2'}


## 6. Start the next session on it

Three shapes come out of an accepted round. Urai history feeds a chat directly — and note the
detour is gone, so only the fossick route survives.

In [ ]:
for m in warm_start(review):
    shows = str(m.get('content')) or ' '.join(c.name for c in m.get('tool_calls', []))
    print(f"{m['role']:9} | {shows[:52]}")

user      | Use fossick to research the AnswerDotAI llmdojo gith
assistant | run_shell
tool      | # llmdojo
LLM coding agents imitate what their conte
assistant | FOSSICK read the repository directly, so its files a


A bootstrap prompt carries the round to a host whose command line takes no prepared history, which
is what Ramabana needs. Tool results are clipped to the 600 characters Ramabana itself keeps on
replay.

In [ ]:
print(bootstrap_prompt(review))

The reviewed Drona round below is the tool route to follow.

User: Use fossick to research the AnswerDotAI llmdojo github repository

Assistant tool: run_shell({"command": "fossick read-gh-repo https://github.com/AnswerDotAI/llmdojo"})

Tool result: # llmdojo
LLM coding agents imitate what their context shows.

Assistant: FOSSICK read the repository directly, so its files are the evidence.

Reply with exactly: DRONA_READY


`drona-start` prints two commands; `--launch` runs them. The first sends the round as one bootstrap
turn, the second resumes the session that turn created.

```sh
drona-start training/llmdojo.ipynb --root /path/to/project --launch
```

In [ ]:
for name, cmd in start_round(review, root='/path/to/project').items(): print(name, cmd[:3])

bootstrap ['ramabana', '--root', '/path/to/project']
resume ['ramabana', '--root', '/path/to/project']


## Move a round between hosts

The notebook is the interchange format, so a session can be captured on one host, reviewed once, and
started on another.

```sh
drona-capture training/round.ipynb                                     # ramabana
drona-capture training/round.ipynb --host claude --cwd /path/to/project
drona-capture training/round.ipynb --host codex  --cwd /path/to/project

drona-export training/round.ipynb ramabana --output training/round.txt
drona-export training/round.ipynb claude --cwd /path/to/project
drona-export training/round.ipynb codex --output training/items.json
```

Claude Code resumes the id the Claude export prints. Codex export is not resumable, because
llmsurgery has no public rollout writer; its items still suit inspection and datasets.

## Develop

```sh
uv sync --all-extras --group dev
uv run nbdev-export
uv run nbdev-test
uv run nbdev-readme
uv run nbdev-clean
```